**Scratchpad** = a session-scoped place to keep intermediate agent state (notes) outside the LLM context window.

**LangGraph provides:** (a) a state object passed between nodes (scratchpad), (b) checkpointing for thread-scoped state, and (c) long-term memory stores for cross-session context.

In [14]:
!pip install langchain langchain_community langchain_core langgraph langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 0.3.75
    Uninstalling langchain-core-0.3.75:
      Successfully uninstalled langchain-core-0.3.75


In [2]:
from typing import TypedDict

from rich.console import Console
from rich.pretty import pprint


console = Console()

In [3]:
# State schema using TypedDict
from typing import TypedDict

class State(TypedDict):
    topic: str
    joke: str

In [4]:
import getpass
import os
from google.colab import userdata
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

In [15]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [16]:
llm = init_chat_model("gpt-4o", model_provider="openai", temperature=0)

**State / StateGraph:** the data schema and directed graph that

moves and mutates that schema across nodes.

**Checkpointer:** saves snapshots of the state per thread so you can resume or inspect past execution.

**Store / BaseStore:** key-value persistence layer (InMemoryStore in examples) for long-term memory.

**Thread:** a single interaction/session id (e.g., thread_id) whose states are checkpointed.

**Tools:** external functions the agent can call. BigTool = vector-indexed tool registry enabling semantic tool selection.

**RAG:** retrieval-augmented generation — combine retrievers + LLMs for context-aware answers.

**MessagesState:** a special state type for message-based agents (system/user/tool messages).


In [17]:
# Simple node that generates a joke using an LLM instance "llm"
def generate_joke(state: State) -> dict[str, str]:
    topic = state["topic"]
    msg = llm.invoke(f"Write a short joke about {topic}")
    return {"joke": msg.content}

# Build and run StateGraph
from langgraph.graph import START, END, StateGraph
workflow = StateGraph(State)
workflow.add_node("generate_joke", generate_joke)
workflow.add_edge(START, "generate_joke")
workflow.add_edge("generate_joke", END)
chain = workflow.compile()

# invoke
result_state = chain.invoke({"topic": "cats"})
print(result_state)  # {'topic': 'cats', 'joke': '...'}

{'topic': 'cats', 'joke': 'Why was the cat sitting on the computer? \n\nBecause it wanted to keep an eye on the mouse!'}


The dictionary returned by the execution result actually represents the joke generation state of the agent. This simple example demonstrates the basic mechanism of how to write contextual information to the state.

We can further explore **checkpointing techniques** for saving and restoring graph state, and human-computer interaction looping techniques for pausing a workflow to obtain human input before continuing.

## Checkpointing + In-memory store (threaded memory)

The LangGraph framework's memory mechanism consists of two key components: checkpointing technology saves the state of the graph at each step of a thread, where a thread has a unique identifier and typically represents a complete interaction (similar to a single conversation in ChatGPT); and long-term memory technology allows specific context to be maintained across threads, whether it is a single file (such as a user profile) or a collection of memories. This system is implemented based on the BaseStore interface, a key-value store that can be used in memory (as shown in this example) or deployed with the LangGraph platform.

In [18]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

checkpointer = InMemorySaver()
memory_store = InMemoryStore()

# memory-aware node
namespace = ("rlm", "joke_generator")

def generate_joke_with_memory(state: State, store: BaseStore) -> dict[str, str]:
    prior = store.get(namespace, "last_joke")
    prior_text = prior.value["joke"] if prior else None

    prompt = (
        f"Write a short joke about {state['topic']}."
        f" Make it different from prior joke: {prior_text or 'None'}"
    )
    msg = llm.invoke(prompt)

    store.put(namespace, "last_joke", {"joke": msg.content})
    return {"joke": msg.content}

# compile with checkpointer and store
workflow = StateGraph(State)
workflow.add_node("generate_joke", generate_joke_with_memory)
workflow.add_edge(START, "generate_joke")
workflow.add_edge("generate_joke", END)
chain = workflow.compile(checkpointer=checkpointer, store=memory_store)

# run in thread 1
config = {"configurable": {"thread_id": "1"}}
chain.invoke({"topic": "cats"}, config)

{'topic': 'cats',
 'joke': 'Why did the cat sit on the computer? \n\nBecause it wanted to keep an eye on the mouse!'}

In [19]:
latest_state = chain.get_state(config)

console.print("\n[bold magenta]Latest graph state (Thread 1):[/bold magenta]")
pprint(latest_state)

Latest graph state (Thread 1):

StateSnapshot(
│   values={
│   │   'topic': 'cats',
│   │   'joke': 'Why did the cat sit on the computer? \n\nBecause it wanted to keep an eye on the mouse!'
│   },
│   next=(),
│   config={
│   │   'configurable': {
│   │   │   'thread_id': '1',
│   │   │   'checkpoint_ns': '',
│   │   │   'checkpoint_id': '1f08f7da-33c7-6d00-8001-cb80af1ec2b7'
│   │   }
│   },
│   metadata={'source': 'loop', 'step': 1, 'parents': {}},
│   created_at='2025-09-12T02:10:22.081946+00:00',
│   parent_config={
│   │   'configurable': {
│   │   │   'thread_id': '1',
│   │   │   'checkpoint_ns': '',
│   │   │   'checkpoint_id': '1f08f7da-2b3a-6d3a-8000-ed634e15ea42'
│   │   }
│   },
│   tasks=(),
│   interrupts=()
)

You can see that the state records the last conversation with the agent, in which we asked it to tell a joke about a cat.

In [20]:
config = {"configurable": {"thread_id": "2"}}
joke_generator_state = chain.invoke({"topic": "cats"}, config)

# Display results, showing cross-thread memory persistence
console.print("\n[bold yellow]Workflow result (Thread 2):[/bold yellow]")
pprint(joke_generator_state)

Workflow result (Thread 2):

{
│   'topic': 'cats',
│   'joke': 'Why did the cat bring a ladder to the bar?\n\nBecause it heard the drinks were on the house!'
}

You can verify that the joke from the first thread was successfully saved to memory.

Read further:  LangMem memory abstraction technology

**Why scratchpad?** Prevents context loss across long workflows; keeps intermediate outputs accessible without bloating prompt.

**Checkpoint vs store:** checkpoint = thread-level snapshots (resume/inspect); store = cross-thread long-term memory (profiles, examples).

**Selection matters:** naive memory retrieval injects irrelevant info. Use embeddings + re-ranking to fetch relevant memories.

**Tool overload problem** → BigTool (semantic tool retrieval) reduces hallucinated or wrong tool calls.

**RAG best practice:** iterative retrieval + re-rank + LLM decision loop until answer confidence is reached.